In [ ]:
!pip install -q avalanche-lib==0.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.4/532.4 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.8/452.8 kB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 16.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 kB 22.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.2 MB/s eta 0:00:00


In [ ]:
import torch
from torch import nn
from torch.optim import SGD
from torch.nn import CrossEntropyLoss
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from avalanche.evaluation.metrics import forgetting_metrics, accuracy_metrics, loss_metrics, timing_metrics, cpu_usage_metrics, confusion_matrix_metrics, disk_usage_metrics
from avalanche.models import SimpleMLP
from avalanche.logging import InteractiveLogger, TextLogger, TensorboardLogger
from avalanche.training.plugins import EvaluationPlugin
from avalanche.training.supervised import Naive

from pathlib import Path
import zipfile
import os
import shutil
import pandas as pd
import numpy as np

In [ ]:
HX_TRAINING_ORIGIN_DIR_PATH = Path('./drive/MyDrive/Colab Notebooks/hx_training_classify')

In [ ]:
DATA_DIR_PATH = Path('./data')
if os.path.exists(DATA_DIR_PATH):
  print(f"{DATA_DIR_PATH} directory already exists")
else:
   DATA_DIR_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
#Copy hx_training_directory to data_dir
HX_TRAINING_DEST_DIR_PATH = os.path.join(DATA_DIR_PATH,"hx_training")
shutil.copytree(HX_TRAINING_ORIGIN_DIR_PATH, HX_TRAINING_DEST_DIR_PATH)

'data/hx_training'

In [ ]:
#Transform CSV file to a DataFrame and a Dictionary
ANNOTATIONS_PATH = Path('./data/hx_training/FSW dataset A_annotations.csv')
df_annotations = pd.read_csv(ANNOTATIONS_PATH, sep=";")

In [ ]:
df_annotations

,#,begin,end,Temperatur,Grat,Nahtinneres,temp_label,grat_label,inner_label,filename,img_path,Weld Seam Nr.,Group
0,0,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-25.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-2...,1,1
1,1,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-14.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-1...,1,1
2,2,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-37.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-3...,1,1
3,3,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-57.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-5...,1,1
4,4,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-18.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-1...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8319,8455,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-52.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8320,8456,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-36.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8321,8457,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-31.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8322,8458,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-44.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7


In [ ]:
df_dictionary = df_annotations.to_dict()

In [ ]:
#Unzip image folder
ZIPFILE_PATH = os.path.join(HX_TRAINING_DEST_DIR_PATH, "hx_training_classify.zip")
with zipfile.ZipFile(ZIPFILE_PATH, 'r') as zip_ref:
  zip_ref.extractall(DATA_DIR_PATH)

In [ ]:
def creationLabelFolders(CREATION_PATH, groups, labels, dir_name):
  MAIN_DIR_PATH = os.path.join(CREATION_PATH, dir_name)
  MAIN_DIR_PATH = Path(MAIN_DIR_PATH)

  path_dictionary = {}
  if os.path.exists(MAIN_DIR_PATH):
    print(f"{dir_name} folder already exists")
  else:
    MAIN_DIR_PATH.mkdir(parents=True, exist_ok=True)

  for group in groups:
    GROUP_DIR_PATH = Path(os.path.join(MAIN_DIR_PATH, f"group_{group}"))
    if os.path.exists(GROUP_DIR_PATH):
      print(f"{GROUP_DIR_PATH} already exists")
    else:
      GROUP_DIR_PATH.mkdir(parents=True, exist_ok=True)
    path_dictionary[f"GROUP_{group}_PATH"] = GROUP_DIR_PATH
    for label in labels:
      LABEL_DIR_PATH = Path(os.path.join(GROUP_DIR_PATH, label))
      if os.path.exists(LABEL_DIR_PATH):
        print(f"{LABEL_DIR_PATH} already exists")
      else:
        LABEL_DIR_PATH.mkdir(parents=True, exist_ok=True)
      #path_dictionary[f"{label}_PATH"] = LABEL_DIR_PATH
  return path_dictionary

In [ ]:
#Create label folders
unique_grat_keys = set(df_dictionary['Grat'].values())
unique_grat_keys_list = list(unique_grat_keys)
print(unique_grat_keys_list)

groups_keys = set(df_dictionary['Group'].values())
groups_keys_list = list(groups_keys)
print(groups_keys_list)

grat_path_dic = creationLabelFolders(DATA_DIR_PATH, groups_keys_list, unique_grat_keys_list, "Grat")

['Grat OK', 'Grat zu stark']
[1, 2, 3, 4, 5, 6, 7]


In [ ]:
grat_path_dic

{'GROUP_1_PATH': PosixPath('data/Grat/group_1'),
 'GROUP_2_PATH': PosixPath('data/Grat/group_2'),
 'GROUP_3_PATH': PosixPath('data/Grat/group_3'),
 'GROUP_4_PATH': PosixPath('data/Grat/group_4'),
 'GROUP_5_PATH': PosixPath('data/Grat/group_5'),
 'GROUP_6_PATH': PosixPath('data/Grat/group_6'),
 'GROUP_7_PATH': PosixPath('data/Grat/group_7')}

In [ ]:
grat_path_list = list(grat_path_dic.values())
grat_path_list

[PosixPath('data/Grat/group_1'),
 PosixPath('data/Grat/group_2'),
 PosixPath('data/Grat/group_3'),
 PosixPath('data/Grat/group_4'),
 PosixPath('data/Grat/group_5'),
 PosixPath('data/Grat/group_6'),
 PosixPath('data/Grat/group_7')]

In [ ]:
for key, value in df_dictionary['filename'].items():
    weld_seam_pos = df_dictionary['Weld Seam Nr.'][key]
    group_pos = df_dictionary['Group'][key]
    filename_pos = df_dictionary['filename'][key]
    file_path = Path(f"./data/hx_training_classify/{weld_seam_pos}/cutouts/{filename_pos}")
    label_pos = df_dictionary['Grat'][key]
    if label_pos == unique_grat_keys_list[0]:
      new_file_path = Path(f"./data/Grat/group_{df_dictionary['Group'][key]}/{unique_grat_keys_list[0]}/{filename_pos}_{weld_seam_pos}.jpg")
      #print(new_file_path)
      shutil.copy2(file_path,new_file_path)
    else:
      new_file_path = Path(f"./data/Grat/group_{df_dictionary['Group'][key]}/{unique_grat_keys_list[1]}/{filename_pos}_{weld_seam_pos}.jpg")
      #print(new_file_path)
      shutil.copy2(file_path,new_file_path)

In [ ]:
data_transform = transforms.Compose([
    transforms.ToTensor()
])

In [ ]:
group_1_dataset = datasets.ImageFolder(root=grat_path_list[0],
                                       transform = data_transform,
                                       target_transform = None)
group_2_dataset = datasets.ImageFolder(root=grat_path_list[1],
                                       transform = data_transform,
                                       target_transform = None)
group_3_dataset = datasets.ImageFolder(root=grat_path_list[2],
                                       transform = data_transform,
                                       target_transform = None)
group_4_dataset = datasets.ImageFolder(root=grat_path_list[3],
                                       transform = data_transform,
                                       target_transform = None)
group_5_dataset = datasets.ImageFolder(root=grat_path_list[4],
                                       transform = data_transform,
                                       target_transform = None)
group_6_dataset = datasets.ImageFolder(root=grat_path_list[5],
                                       transform = data_transform,
                                       target_transform = None)
group_7_dataset = datasets.ImageFolder(root=grat_path_list[6],
                                       transform = data_transform,
                                       target_transform = None)
dataset_arr = [group_1_dataset, group_2_dataset, group_3_dataset,group_4_dataset,group_5_dataset,group_6_dataset,group_7_dataset]

In [ ]:
len(dataset_arr[0])

950

In [ ]:
#Split datasets
splitted_datasets={"group_1_dataset":{},"group_2_dataset":{},"group_3_dataset":{},"group_4_dataset":{},"group_5_dataset":{},"group_6_dataset":{},"group_7_dataset":{}}

for i in range(0,7):
  train_size = int(0.8 * len(dataset_arr[i]))
  test_size = len(dataset_arr[i]) - train_size
  #j = 1
  splitted_datasets[f"group_{i+1}_dataset"]["train_dataset"], splitted_datasets[f"group_{i+1}_dataset"]["test_dataset"] = torch.utils.data.random_split(dataset_arr[i], [train_size, test_size])
  #j = j+1
  #print(j)

splitted_datasets

{'group_1_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7869367b3400>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7869367b3430>},
 'group_2_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7869367b33a0>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7869367b3130>},
 'group_3_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7869367b0e20>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7869367b1840>},
 'group_4_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7869367b1c60>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7869367b0f70>},
 'group_5_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7869367b18d0>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7869367b2ad0>},
 'group_6_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7869367b2b60>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7869367b2c50>},
 'group_7_dataset': {'